In [ ]:
import os
import shutil
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [ ]:
ref_labels_dir = "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/fused/labels/" 

num_classes = 20

modalidades_dirs = {
    "FUSED": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/fused/labels/",
    "FUSED_NDVI": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/fused_ndvi/labels/",
    "FUSED_NDRE": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/fused_ndre/labels/",
    "NDVI": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/ndvi/labels/",
    "NDRE": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/ndre/labels/",
    "RGB": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/rgb/labels/",
    "RGB_NDVI": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/rgb_ndvi/labels/",
    "RGB_NDRE": "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/fotos-rotuladas/rgb_ndre/labels/"
}

output_dir = "/mnt/sdb-seagate/graduacao/datasets/projeto_cerrado/dataset/"

In [ ]:
train_ratio = 0.70
val_ratio = 0.20
test_ratio = 0.10

images_list = []
labels_matrix = []

In [ ]:
for txt_file in os.listdir(ref_labels_dir):
    if not txt_file.endswith(".txt"):
        continue
    
    base_name = txt_file.replace(".txt", "") 
    filepath = os.path.join(ref_labels_dir, txt_file)

    class_presence = np.zeros(num_classes)
    
    with open(filepath, 'r') as f:
        linhas = f.readlines()
        for linha in linhas:
            linha = linha.strip()
            if linha:
                class_id = int(linha.split()[0])
                if class_id < num_classes:
                    class_presence[class_id] = 1 
            
    images_list.append(base_name) 
    labels_matrix.append(class_presence)

X = np.array(images_list)
Y = np.array(labels_matrix)

In [ ]:
print(f"Total de imagens válidas para divisão: {len(X)}")

msss_train = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=(val_ratio + test_ratio), random_state=42)
train_idx, rest_idx = next(msss_train.split(X, Y))

X_train, Y_train = X[train_idx], Y[train_idx]
X_rest, Y_rest = X[rest_idx], Y[rest_idx]

# Segundo split: Validação vs Teste
test_fraction = test_ratio / (val_ratio + test_ratio) 
msss_val = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_fraction, random_state=42)
val_idx, test_idx = next(msss_val.split(X_rest, Y_rest))

X_val = X_rest[val_idx]
X_test = X_rest[test_idx]

print(f"Divisão calculada -> Treino: {len(X_train)} | Val: {len(X_val)} | Teste: {len(X_test)}")

In [ ]:
def copy_files(file_list, split_name):
    extensoes_possiveis = [".jpg", ".JPG", ".tif", ".TIF", ".png", ".PNG"]
    
    for mod_name, mod_path in modalidades_dirs.items():
        dest_img_dir = os.path.join(output_dir, mod_name, "images", split_name)
        dest_lbl_dir = os.path.join(output_dir, mod_name, "labels", split_name)
        os.makedirs(dest_img_dir, exist_ok=True)
        os.makedirs(dest_lbl_dir, exist_ok=True)
        
        for base_name in file_list: # Agora a file_list só tem nomes bases
            txt_file = base_name + ".txt"
            src_txt = os.path.join(mod_path, "labels", txt_file)
            
            # 1. Copia o arquivo .txt (esse a gente tem certeza que existe e é .txt)
            if os.path.exists(src_txt):
                shutil.copy(src_txt, os.path.join(dest_lbl_dir, txt_file))
                
            # 2. Caça a imagem correspondente tentando todas as extensões
            imagem_encontrada = False
            for ext in extensoes_possiveis:
                img_file = base_name + ext
                src_img = os.path.join(mod_path, "images", img_file)
                
                if os.path.exists(src_img):
                    # Achou! Copia e para de procurar as outras extensões
                    shutil.copy(src_img, os.path.join(dest_img_dir, img_file))
                    imagem_encontrada = True
                    break 
            
            if not imagem_encontrada:
                print(f"⚠️ AVISO: Imagem não encontrada para o label {txt_file} na pasta {mod_name}")

In [ ]:
print("\nCopiando imagens de Treino...")
copy_files(X_train, "train")
print("Copiando imagens de Validação...")
copy_files(X_val, "val")
print("Copiando imagens de Teste...")
copy_files(X_test, "test")

print("\n🎉 Divisão estratificada multirrótulo concluída com sucesso!")
print(f"Seus dados estão prontos na pasta: {output_dir}")